# Hybrid RAG Building 

In [1]:
import warnings
warnings.filterwarnings('ignore')

from langchain_community.document_loaders import PyPDFLoader
pdf = PyPDFLoader("Why_Language_Models_Hallucinate_Explainer.pdf")
document = pdf.load()

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter 
text_splitter = RecursiveCharacterTextSplitter(chunk_overlap=120,chunk_size=1200)
pages = text_splitter.split_documents(documents=document)
chunks = [i.page_content for i in pages]
len(chunks)

10

In [3]:
import chromadb 
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction 
embedding_function = SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

client = chromadb.PersistentClient(path="./Local_data")

collection = client.get_or_create_collection(
    name="edumind",
    embedding_function=embedding_function
)

if collection.count() == 0:
    collection.add(
        documents=chunks,
        ids=[str(i) for i in range(len(chunks))],
        metadatas=[i.metadata for i in pages]
    )

print(collection.count())

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10385.19it/s]


10


In [4]:
from rank_bm25 import BM25Okapi 
def fun(token):
    token = token.lower()
    token = token.split()
    return token 
from rank_bm25 import BM25Okapi 
token = [fun(i) for i in chunks]
token_cor = BM25Okapi(token)

In [5]:
from langchain_groq import ChatGroq 
import os 
from dotenv import load_dotenv 
load_dotenv()
key = os.getenv("GROQ_API_KEY")
chat_model = ChatGroq(model="openai/gpt-oss-120b")

In [6]:
def Hybrid_search(query:str)->str:
    prompt = f""" for symentic search only : {query}"""
    query_rewrite = chat_model.invoke(prompt).content
    # query_rewrite = query_rewrite[:1000]

    result = collection.query(query_texts=[query_rewrite],n_results=3)
    distance = result['distances'][0]
    document = result['documents'][0]
    threshold = 0.8
    near_chunks = []
    print(distance)

    for dist, doc in zip(distance, document):
        if dist < threshold:
            near_chunks.append(doc)

    score = token_cor.get_scores(query_rewrite.split())
    def get_scores(score , k=10):
        index  = list(enumerate(score))
        sorted_index = sorted(index , key = lambda x:x[1], reverse=True)
        return [doc for doc , i in sorted_index[:k]]

    get_top_scores = get_scores(score , k=10)
    copy_top_scores = [chunks[i] for i in get_top_scores]

    rrf_token={}

    for rank , doc in enumerate(near_chunks):
        rrf_token[doc]=rrf_token.get(doc,0)+1/(rank+60)

    for rank , doc in enumerate(copy_top_scores):
        rrf_token[doc]=rrf_token.get(doc,0)+1/(rank+60)
    marge = sorted(rrf_token.items() , key=lambda x:x[1] , reverse=True)
    hybrid_top_docs=[doc for doc, _ in marge[:5]]
    if near_chunks:
        return {
    "type": "rag",
    "content": "\n\n".join(hybrid_top_docs)
}

    from tavily import TavilyClient

    client = TavilyClient(
        api_key=os.getenv("TAVILY_API_KEY")
    )
    response = client.search(query=query_rewrite)
    return {
    "type": "web",
    "content": str(response)
}

In [15]:
questions = [
    "What is hallucination mean by LLM ?",
    "What is the largest country in the world?",
    "What is the GDP of India?"
]

for question in questions:

    # result = Research_tool(question)
    result = Hybrid_search(question)

    if result["type"] == "rag":
        source = "Research Documents"
    else:
        source = "Tavily Web Search"

    print("\nQuestion:", question)
    print("Answer:", result)
    print("Sources:", result["type"])

[0.2524198293685913, 0.30057328939437866, 0.32228779792785645]

Question: What is hallucination mean by LLM ?
Answer: {'type': 'rag', 'content': "1\n Why Language Models Hallucinate: A Statistical Account\n A Summary and Synthesis of Current Research  |  Compiled 2026\nABSTRACT\nHallucination — the tendency of large language models (LLMs) to produce fluent, confident, and factually\nwrong statements — is often described as mysterious or unpredictable. Recent theoretical work, most\nnotably by Kalai, Nachum, Vempala, and Zhang (2025), argues the opposite: hallucination is a natural and\nlargely predictable consequence of how models are trained and graded, not an unexplained quirk of scale.\nThis document synthesizes that argument alongside supporting survey literature, covering the pretraining\norigins of factual error, why post-training and benchmarking fail to eliminate it, the specific error categories\nthat resist correction regardless of model size, and the mitigation strategies — 